In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer

from benchmark.config import (
    DATA_PROCESSED,
    DATASETS,
    MODELS_DIR,
    N_TRIALS,
    RESULTS_BENCHMARK,
)
from benchmark.models.random_forest_model import fit_random_forest_cross_validation
from benchmark.models.random_forest_model import fit_best_random_forest

/home/shado/college_project/comp9417/9417GroupProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_name = "superconductor"
cfg = DATASETS[data_name]
processed = DATA_PROCESSED / data_name

train_df = pd.read_csv(processed / "train.csv")
test_df = pd.read_csv(processed / "test.csv")

with open(processed / "metadata.json", "r") as f:
    metadata = json.load(f)
print(metadata)

{'num_cols': ['number_of_elements', 'mean_atomic_mass', 'wtd_mean_atomic_mass', 'gmean_atomic_mass', 'wtd_gmean_atomic_mass', 'entropy_atomic_mass', 'wtd_entropy_atomic_mass', 'range_atomic_mass', 'wtd_range_atomic_mass', 'std_atomic_mass', 'wtd_std_atomic_mass', 'mean_fie', 'wtd_mean_fie', 'gmean_fie', 'wtd_gmean_fie', 'entropy_fie', 'wtd_entropy_fie', 'range_fie', 'wtd_range_fie', 'std_fie', 'wtd_std_fie', 'mean_atomic_radius', 'wtd_mean_atomic_radius', 'gmean_atomic_radius', 'wtd_gmean_atomic_radius', 'entropy_atomic_radius', 'wtd_entropy_atomic_radius', 'range_atomic_radius', 'wtd_range_atomic_radius', 'std_atomic_radius', 'wtd_std_atomic_radius', 'mean_Density', 'wtd_mean_Density', 'gmean_Density', 'wtd_gmean_Density', 'entropy_Density', 'wtd_entropy_Density', 'range_Density', 'wtd_range_Density', 'std_Density', 'wtd_std_Density', 'mean_ElectronAffinity', 'wtd_mean_ElectronAffinity', 'gmean_ElectronAffinity', 'wtd_gmean_ElectronAffinity', 'entropy_ElectronAffinity', 'wtd_entropy_E

In [3]:
target_column = metadata["target_col"]
x_train = train_df.drop(columns=[target_column]).copy()
y_train = train_df[target_column].to_numpy(dtype=np.float32)

x_test = test_df.drop(columns=[target_column]).copy()
y_test = test_df[target_column].to_numpy(dtype=np.float32)

In [4]:
num_cols = metadata["num_cols"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
    ],
    remainder="passthrough",
)

In [5]:
best_trial = fit_random_forest_cross_validation(
    X=x_train,
    y=y_train,
    processor=preprocessor,
    task=cfg["task"],
    n_trials=N_TRIALS,
)

rf_model, rf_pipeline, rf_fit_time = fit_best_random_forest(
    X=x_train,
    y=y_train,
    best_trial=best_trial,
    processor=preprocessor,
    task=cfg["task"],
)

[I 2026-04-20 19:14:14,689] A new study created in memory with name: no-name-938bd2f3-4ae2-4756-a380-37e715198e42
[I 2026-04-20 19:14:15,443] Trial 0 finished with value: 157.13987973443508 and parameters: {'n_estimators': 60, 'max_depth': 9, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'log2', 'bootstrap': True}. Best is trial 0 with value: 157.13987973443508.
[I 2026-04-20 19:14:16,372] Trial 1 finished with value: 146.86526436351045 and parameters: {'n_estimators': 49, 'max_depth': 9, 'min_samples_split': 7, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 1 with value: 146.86526436351045.
[I 2026-04-20 19:14:17,010] Trial 2 finished with value: 201.75647150443106 and parameters: {'n_estimators': 50, 'max_depth': 7, 'min_samples_split': 11, 'min_samples_leaf': 5, 'max_features': 'log2', 'bootstrap': True}. Best is trial 1 with value: 146.86526436351045.
[I 2026-04-20 19:14:17,651] Trial 3 finished with value: 179.1419579207977 and pa

In [6]:
import time
from benchmark.metrics import evaluate_regression


x_test_transformed = rf_pipeline.transform(x_test)

t0 = time.perf_counter()
y_test_pred = rf_model.predict(x_test_transformed)
inference_time = time.perf_counter() - t0

inference_time_per_sample = inference_time / len(x_test)

test_metrics = evaluate_regression(y_test, y_test_pred)


print("Best params:", best_trial.params)
print("Training time (s):", rf_fit_time)
print("Inference time per sample (s):", inference_time_per_sample)
print("Test metrics:", test_metrics)

Best params: {'n_estimators': 62, 'max_depth': 12, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'bootstrap': True}
Training time (s): 0.21840597400023398
Inference time per sample (s): 6.4467756878662905e-06
Test metrics: {'rmse': 10.7205}


In [7]:
models_dir = MODELS_DIR / data_name
models_dir.mkdir(parents=True, exist_ok=True)

results_dir = RESULTS_BENCHMARK / data_name
results_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(rf_model, models_dir / "random_forest.joblib")
joblib.dump(rf_pipeline, models_dir / "random_forest_pipeline.joblib")

output = {
    **test_metrics,
    "fit_time_s": float(rf_fit_time),
    "infer_time_per_sample_s": float(inference_time_per_sample),
    "best_params": best_trial.params,
}

with open(results_dir / "random_forest.json", "w") as f:
    json.dump(output, f, indent=4)

print(results_dir / "random_forest.json")

/home/shado/college_project/comp9417/9417GroupProject/results/superconductor/random_forest.json
